# 预报时效拆分

包名 `time_split_linear`。三个插件只处理时间维：把粗时效插成细时效，或把多个时次沿时间拼接。空间网格不改。

在本包目录下使用。命令行没写的参数先读 `resource/conf.ini`，配置里也没有再用代码预设。命令行只整理并打印参数，不读预报文件。格点读写在插件的 `process` 里。

时效三元组是 `[起点, 终点, 步长]`，终点包含在结果中。覆盖 0 到 72 小时时，3 小时输入写 `0,72,3`，1 小时输出写 `0,72,1`。


## TimeSplit

把同一起报时间的粗时效文件沿 `dtime` 插值到更密的时效。默认 `slinear`：已有时效保持原值，中间时效落在相邻两帧的连线上。

适用于逐 3 小时一帧收成逐小时一帧。输入若是从起报累计到该时效的序列，插值结果是逐小时累计；再设 `delta_hour=1`，得到逐小时增量，增量把这一段的变化平均分到每个小时。近 3 小时合计要先加回从起报开始的累计，再走这两步。

给出 `uv_fmt` 时，`input_fmt` 作为 U，`uv_fmt` 作为 V，插值前只保留风速。给出 `output_fmt` 时，每个输出时效写一个 nc，小于 0 的值改为 0。

### 构造参数

| 参数 | 含义 | 默认 |
|---|---|---|
| `input_fh_list` | 输入时效 `[起点, 终点, 步长]`，终点包含在结果中 | 必填 |
| `output_fh_list` | 输出时效 `[起点, 终点, 步长]`，终点包含在结果中 | 必填 |
| `interp_method` | 传给 `DataArray.interp` 的方法 | `slinear` |
| `output_fmt` | 输出路径模板；空则不写文件 | `None` |
| `uv_fmt` | V 分量路径模板；空则不合成风速 | `None` |
| `delta_hour` | 沿时效做差的间隔；空则不做差分 | `None` |
| `is_fill0` | 为真时，时效 0 不读文件，改用全 0 场 | `False` |
| `io_method` | 读网格函数；空则用 `read_griddata_from_nc` | `None` |

### `process` 参数

| 参数 | 含义 |
|---|---|
| `input_fmt` | 输入路径模板，可含 `YYYY`、`MM`、`DD`、`HH`、`TTT` |
| `time` | 起报时间，需能被 `strftime` 解析 |

返回 `( 细时效场, 插值前的粗时效场 )`。读数或插值失败时返回 `(None, None)`。

拼接只有一个起报时间的文件时，结果里会多出一个 12 小时后的起报时间，值为缺测 `999999`。

### 插件示例

```python
from datetime import datetime
from src.cal_griddata_TimeSplit_plugin import TimeSplit

# 逐 3 小时插成逐小时，0 到 72 小时
splitter = TimeSplit(
    input_fh_list=[0, 72, 3],
    output_fh_list=[0, 72, 1],
    interp_method="slinear",
    is_fill0=False,
)
fine, raw = splitter.process(
    "fcst/YYYYMMDDHH.TTT.nc",
    datetime(2023, 7, 1, 0),
)

# 从起报累计的序列：先插成逐小时累计，再得到逐小时增量
increment = TimeSplit(
    input_fh_list=[0, 72, 3],
    output_fh_list=[0, 72, 1],
    delta_hour=1,
)
hourly_increment, _ = increment.process(
    "rain/YYYYMMDDHH.TTT.nc",
    datetime(2023, 7, 1, 0),
)
```

### 命令行

```bash
python -m cli.cal_griddata_TimeSplit_plugin_cli --input-fh 0,72,3 --output-fh 0,72,1 --input-fmt fcst/YYYYMMDDHH.TTT.nc --time 2023070100
python cli/cal_griddata_TimeSplit_plugin_cli.py --input-fh 0,72,3 --output-fh 0,72,1 --interp-method linear --delta-hour 1 --input-fmt rain/YYYYMMDDHH.TTT.nc --time 2023070100
```

## Timesplit_hourtomin

把已经在内存里的小时场插到更细的时间轴。适用于逐小时一帧收成逐 10 分钟或逐 30 分钟。默认间隔 10 分钟，插值方法 `slinear`。不写文件，返回一块包含全部细时间的场。

`index="dtime"` 时，输入的预报时效单位是小时。插值前在最前面补一个时效为 0 的场；没有传入 `fcst0` 时，这个场是全 0。分钟输出会先把小时坐标乘以 60，再按 `ntime` 分钟插值。`index="time"` 沿起报时间插值，不补零场。

### 构造参数

| 参数 | 含义 | 默认 |
|---|---|---|
| `ntime` | 插值后的时间间隔 | `10` |
| `is_Min` | 为真时单位是分钟，为假时单位是小时 | `True` |
| `index` | 插值轴：`dtime` 预报时效，`time` 起报时间 | `dtime` |
| `interp_method` | 传给 `DataArray.interp` 的方法 | `slinear` |

### `process` 参数

| 参数 | 含义 |
|---|---|
| `fcst` | 已读入内存的格点场。沿 `dtime` 插值时，时效单位为小时 |
| `fcst0` | 时效为 0 的场。为空则按第一帧造全 0 场；只在 `index="dtime"` 时使用 |

返回插值后的 `DataArray`。`index` 不是 `dtime` 或 `time`，或零场与第一帧形状不一致时，返回 `None`。

### 插件示例

```python
from src.timesplit_hourtomin_plugin import Timesplit_hourtomin

# 逐小时预报时效收成逐 30 分钟
to_30min = Timesplit_hourtomin(
    ntime=30,
    is_Min=True,
    index="dtime",
    interp_method="slinear",
)
minute_fcst = to_30min.process(hourly_fcst, fcst0=None)

# 逐 10 分钟；沿起报时间插值时不补零场
to_10min = Timesplit_hourtomin(ntime=10, is_Min=True, index="time")
minute_by_issue = to_10min.process(hourly_by_issue_time)
```

### 命令行

```bash
python -m cli.timesplit_hourtomin_plugin_cli --ntime 30 --index dtime --interp-method slinear
python cli/timesplit_hourtomin_plugin_cli.py --ntime 10 --no-is-min --index time
```

## Readdata_fromtime

按同一路径模板读取多个时次的 nc，裁到指定经纬度范围，再沿时间堆成六维场。只拼接，不做时间插值，也不改变每个文件里原有的预报时效间隔。

返回维度为 `member, time, dtime, level, lat, lon`。`time` 使用传入的时间列表，成员、时效和层次来自第一个文件。

`process` 的定义没有 `self`，按函数方式调用。经纬度坐标依赖包外的 `dlonlat_info`。

### `process` 参数

| 参数 | 含义 |
|---|---|
| `file_path` | 路径模板，由 `meteva_base.get_path` 按时间展开 |
| `file_time` | 要读取的时间列表 |
| `glon` | 经度 `[ 起点, 终点, 分辨率 ]` |
| `glat` | 纬度 `[ 起点, 终点, 分辨率 ]` |

### 插件示例

```python
from datetime import datetime
from src.readdata_fromtime_plugin import Readdata_fromtime

data = Readdata_fromtime.process(
    "fcst/YYYYMMDDHH.nc",
    [datetime(2023, 7, 1, 0), datetime(2023, 7, 1, 1)],
    [70, 140, 0.05],
    [0, 60, 0.05],
)
```

### 命令行

```bash
python -m cli.readdata_fromtime_plugin_cli --file-path fcst/YYYYMMDDHH.nc --file-time 2023070100,2023070101 --glon 70,140,0.05 --glat 0,60,0.05
python cli/readdata_fromtime_plugin_cli.py --file-path fcst/YYYYMMDDHH.nc --file-time 2023070100,2023070101 --glon 70,140,0.05 --glat 0,60,0.05
```

## 命令行公共规则

在本包目录执行。每条命令都可以用 `python -m cli.<模块>_cli`，也可以直接运行 `python cli/<模块>_cli.py`。

参数优先级是：命令行显式传入，然后是 `resource/conf.ini` 里的非空项，再是代码预设。`--config` 可以改读另一个 ini，相对路径按本包目录解析。

`input_fh`、`output_fh`、`input_fmt`、`time`，以及读数拼接用的 `file_path`、`file_time`、`glon`、`glat`，命令行和配置都空着时会报错。`hourtomin` 的间隔、单位、插值轴和插值方法都有预设，可以不写。

下面三条命令只确认参数，不读取预报文件。

```bash
python -m cli.cal_griddata_TimeSplit_plugin_cli --input-fh 0,72,3 --output-fh 0,72,1 --input-fmt fcst/YYYYMMDDHH.TTT.nc --time 2023070100 --uv-fmt windv/YYYYMMDDHH.TTT.nc --output-fmt out/YYYYMMDDHH.TTT.nc
python -m cli.timesplit_hourtomin_plugin_cli --ntime 10 --is-min --index dtime --interp-method slinear
python -m cli.readdata_fromtime_plugin_cli --file-path fcst/YYYYMMDDHH.nc --file-time 2023070100,2023070101 --glon 70,140,0.05 --glat 0,60,0.05
```